In [37]:
from typing import TypedDict, Annotated

from dotenv import load_dotenv

load_dotenv()

True

In [38]:
from langchain.chat_models import init_chat_model

In [39]:
from google import genai
from pprint import pprint

client = genai.Client()

m = next(client.models.list())
print(type(m))
#pprint(m.model_dump())  # pydantic: shows all available fields

<class 'google.genai.types.Model'>


In [40]:
llm = init_chat_model("google_genai:gemini-2.5-flash")
print(llm.invoke("Who was the first person to walk on the moon?").content)

The first person to walk on the moon was **Neil Armstrong**.


In [41]:
from langgraph.graph import StateGraph, START, END, add_messages

llm = init_chat_model("google_genai:gemini-2.5-flash")

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return{"messages": [llm.invoke(state["messages"])]}



builder = StateGraph(State)

builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [44]:
message = {"role": "user", "content": "Who is the richest woman on the earth? Print only the name"}
response = graph.invoke({"messages":[message]})

response["messages"]

[HumanMessage(content='Who is the richest woman on the earth? Print only the name', additional_kwargs={}, response_metadata={}, id='939fe2dc-1d34-4491-89d4-8de8a0c3fd11'),
 AIMessage(content='Françoise Bettencourt Meyers', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019caf70-0aa8-7211-8249-59e3da1d34c8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 168, 'total_tokens': 182, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 161}})]

In [46]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)

Bot: The first man on the moon was **Neil Armstrong**.

He landed on the moon with the Apollo 11 mission on July 20, 1969, and was famously the first to step out onto the lunar surface.


KeyboardInterrupt: Interrupted by user